# 第12章 对抗攻击
## Adversarial Attack — 攻击与防御的博弈

**来源：李宏毅《深度学习教程》第12章 | 对应原书第212-226页**

---

## 一、知识地图：全章结构与脉络

```
第12章 对抗攻击
├── 12.1 对抗攻击简介
│   ├── 动机：神经网络需要抵抗恶意攻击
│   ├── 无目标攻击 vs 有目标攻击
│   └── 人眼不可见的噪声改变分类结果
├── 12.2 如何进行网络攻击
│   ├── 优化问题形式化
│   ├── 损失函数设计
│   ├── 约束：d(x, x0) <= epsilon
│   ├── L2范数 vs L-无穷范数（为什么L-无穷更符合人类感知）
│   └── 投影梯度下降
├── 12.3 快速梯度符号法 (FGSM)
│   ├── 只更新一次：sign(grad L) * epsilon
│   └── 迭代版FGSM
├── 12.4 白盒攻击与黑盒攻击
│   ├── 白盒：知道模型参数
│   ├── 黑盒：不知道参数但可查询
│   ├── 代理网络方法
│   └── 集成攻击
├── 12.5 其他模态的攻击
├── 12.6 现实世界中的攻击
│   ├── 神奇眼镜骗过人脸识别
│   ├── 交通标志牌攻击
│   ├── 对抗性重编程
│   └── 模型后门（训练阶段攻击）
├── 12.7 被动防御
│   ├── 平滑模糊化
│   ├── 图像压缩
│   └── 随机化防御
└── 12.8 主动防御（对抗训练）
```

## 二、对抗攻击简介：神经网络比你想象的脆弱

### 2.1 动机

我们将各种高正确率的神经网络部署到真实世界。但仅仅**正确率高**是不够的——模型需要能抵抗来自外界的**恶意攻击**。

真实威胁场景：
- 垃圾邮件发送者会想办法绕过滤器
- 攻击者尝试欺骗人脸识别系统
- 自动驾驶需要正确识别被篡改的交通标志

### 2.2 什么是对抗攻击？

在一张正常图片上加入**人眼完全看不出**的微小噪声 → 网络输出完全错误的结果。

**真实案例（50层ResNet）**：
- 原始图片：虎斑猫，置信度64%
- 加微小噪声后：海星，置信度**100%**！

**两种攻击模式**：
- **无目标攻击**：只要输出不是正确类别就算成功
- **有目标攻击**：输出必须是攻击者指定的类别（如"必须是海星"，更难）

## 三、攻击的数学形式化与实现

### 3.1 作为约束优化问题

**无目标攻击**：
$$\min_x L(x) = -\text{CrossEntropy}(f(x), \hat{y})$$
$$\text{s.t. } d(x_0, x) \leq \varepsilon$$

**有目标攻击**：
$$\min_x L(x) = -\text{CE}(f(x), \hat{y}) + \text{CE}(f(x), y_{target})$$

### 3.2 L2 vs L-无穷：哪个更符合人类感知？

考虑4像素图片的两个版本：
- 情况A：4个像素各改0.5（L2=1, L-无穷=0.5）
- 情况B：3个像素不变，1个改2.0（L2=2, L-无穷=2.0）

人类视觉对单个像素的大幅改变比全局均匀变化更敏感 → L-无穷更符合人类感知。攻击时需要保证L-无穷小，而不只是L2小。

### 3.3 FGSM（快速梯度符号法）

$$x_{adv} = x_0 + \varepsilon \cdot \text{sign}(\nabla_x L(x_0))$$

只需一次梯度计算，每个像素向梯度符号方向移动 $\varepsilon$。这是最简单的攻击方法。

### 3.4 PGD（投影梯度下降，迭代版）

```
x = x0
for t in 1..T:
    x = x + alpha * sign(grad_x L(x))   # 小步更新
    x = x0 + clip(x - x0, -eps, eps)     # 投影回eps球
```

比FGSM更强——多步迭代能更精确地找到攻击方向。

In [ ]:
# ============================================
# PyTorch示例1：FGSM和PGD攻击
# ============================================
import torch
import torch.nn as nn

def fgsm_attack(model, x, y_true, epsilon=0.03):
    """FGSM：一步攻击"""
    x.requires_grad = True
    loss = nn.CrossEntropyLoss()(model(x), y_true)
    loss.backward()
    x_adv = x + epsilon * x.grad.sign()
    return torch.clamp(x_adv, 0, 1).detach()

def pgd_attack(model, x, y_true, epsilon=0.03, alpha=0.01, steps=10):
    """PGD：迭代攻击，更强"""
    x_adv = x.clone().detach()
    x_orig = x.clone().detach()
    for _ in range(steps):
        x_adv.requires_grad = True
        loss = nn.CrossEntropyLoss()(model(x_adv), y_true)
        loss.backward()
        with torch.no_grad():
            x_adv = x_adv + alpha * x_adv.grad.sign()
            eta = torch.clamp(x_adv - x_orig, -epsilon, epsilon)
            x_adv = torch.clamp(x_orig + eta, 0, 1)
    return x_adv.detach()

print("FGSM和PGD攻击已实现。")
print("FGSM: x_adv = x + eps * sign(grad)")
print("PGD: 多步小更新+投影，攻击力更强")

## 四、白盒攻击 vs 黑盒攻击

### 4.1 白盒攻击
攻击者知道模型架构、参数和梯度 → 可直接计算最优扰动。理论上最危险，但现实中难以获取这些信息。

### 4.2 黑盒攻击
攻击者不知道模型内部，但能**查询模型**（给输入得输出）。

**方法**：
1. 用相同训练数据训练一个**代理网络**
2. 对代理网络做白盒攻击，生成对抗样本
3. 对抗样本往往也能攻击目标网络

**实验数据**：黑盒攻击可使目标模型正确率降至50%以下。使用集成攻击（多个代理网络同时攻击），正确率甚至低于6%！

### 4.3 为什么攻击能成功？

实验观察：在小丑鱼图片的"攻击方向"上，所有网络的"安全识别区域"都非常窄。不同网络在这个方向上有惊人的一致性。

**结论**：攻击成功主要源于**数据本身的局限**——有限数据上学到的决策边界在某些方向上天生脆弱。更多的数据可能缓解这个问题。

## 五、极限攻击与现实世界攻击

### 5.1 单像素攻击
只改1个像素！茶壶→摇杆。成功率不高，但证明模型非常脆弱。

### 5.2 通用对抗攻击
一个**固定扰动**对所有图片有效。这意味着攻击者可在摄像头上贴一个固定图案，让所有看到的物体都被错误分类。

### 5.3 对抗性重编程
**寄生**在已有模型上做新任务：将"数方块"任务嵌入在噪声中，ImageNet模型输出"金鱼"（2个方块）、"噬人鲨"（3个方块）等。

### 5.4 模型后门（训练阶段攻击）
在训练数据中嵌入"触发模式"——正常训练，正常分类，但遇到特定触发时输出攻击者预设的结果。非常隐蔽。

### 5.5 物理世界攻击
- **神奇眼镜**：戴上后从所有角度被识别为某知名女艺人
- **交通标志**：STOP上贴纸→识别为限速；拉长"35"→人眼看35，机器看85
- 需考虑：多角度、分辨率、颜色差异

## 六、防御方式

### 6.1 被动防御（不修改模型）
在模型前加"滤波器"：
- 轻度平滑/模糊化 → 破坏攻击噪声的精细结构
- JPEG压缩 → 压缩失真对攻击信号影响大
- 生成器重建 → 生成器没见过攻击噪声

**致命弱点**：一旦攻击者知道防御方式，可将其纳入攻击过程（当作网络第一层），防御失效。

**随机化防御**：随机选择多种防御方式组合，增加攻击者预测难度——但仍可能被通用攻击突破。

### 6.2 主动防御：对抗训练
训练时加入对抗样本：
1. 用正常数据训练模型
2. 用当前模型攻击自己的训练数据，生成对抗样本
3. 对抗样本标注为正确类别
4. 混合原始数据+对抗样本继续训练
5. 循环2-4

可视为数据增强。局限：挡不住未见过的攻击方法；计算开销大。

> **总结**：攻击比防御容易得多——这是AI安全领域的核心挑战。黑盒攻击可行，防御总有漏洞。

In [ ]:
# ============================================
# PyTorch示例2：对抗训练
# ============================================
def adversarial_training_step(model, x, y_true, optimizer, epsilon=0.03, alpha=0.01, steps=5):
    """
    对抗训练：在攻防中提升模型鲁棒性
    1. 用当前模型攻击自己 -> 生成对抗样本
    2. 用对抗样本（保持正确标签）训练模型
    """
    model.eval()
    x_adv = pgd_attack(model, x, y_true, epsilon, alpha, steps)
    
    model.train()
    outputs = model(x_adv)
    loss = nn.CrossEntropyLoss()(outputs, y_true)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    return loss.item()

print("对抗训练已实现。")
print("核心：用模型攻击自己，再用攻击结果训练。")

## 七、核心要点总结

1. 神经网络对人眼不可见的对抗噪声极其脆弱。
2. 攻击是约束优化问题：在距离约束下最大化分类误差。
3. L-无穷范数比L2更符合人类视觉感知。
4. FGSM（一步）和PGD（迭代）是最经典的攻击方法。
5. 黑盒攻击同样危险——对抗方向跨模型可迁移。
6. 攻击成功源于数据局限，不完全是模型问题。
7. 物理世界攻击需考虑多角度、分辨率、色彩等现实因素。
8. 模型后门攻击在训练阶段植入，极其隐蔽。
9. 被动防御可能被绕过；对抗训练（主动防御）最有效但仍有局限。
10. 攻击vs防御是持续的军备竞赛——永远没有完美防御。